# Workflow_1

Notebook backing this PyNode.

In [ ]:
def Workflow_1_function():
    def update_subflow(subflow_nodes_df, **kwargs):
        for idx, row in subflow_nodes_df.iterrows():
            if row.get('node_type') == 'data_node' and row.get('node_is_from_master') == True:
                node_name = str(row.get('node_name', '')).strip()
                if node_name in kwargs:
                    subflow_nodes_df.at[idx, 'node_input_value'] = repr(kwargs[node_name])
        return subflow_nodes_df

    def run_subflow(subflow_name, subflow_nodes_df, subflow_connections_df, python_executable=None):
        f = flow(flow_name=subflow_name, nodes_df=subflow_nodes_df, connections_df=subflow_connections_df)
        f.set_inputs()
        f.get_outputs(key=None)
        f.make()
        append_lines = []
        append_lines.append('import json')
        append_lines.append('_quest_subflow_results = {}')
        append_lines.append("print('__QUEST_SUBFLOW_RESULTS_START__')")
        append_lines.append("print(json.dumps(_quest_subflow_results, default=str))")
        append_lines.append("print('__QUEST_SUBFLOW_RESULTS_END__')")
        f.main_py = f.main_py.rstrip() + '\n\n' + '\n'.join(append_lines) + '\n'
        with tempfile.TemporaryDirectory() as tmpdir:
            f.save(tmpdir + os.sep)
            result = f.run(python_executable=python_executable)
            stdout = getattr(result, 'stdout', '') or ''
        start_marker = '__QUEST_SUBFLOW_RESULTS_START__'
        end_marker = '__QUEST_SUBFLOW_RESULTS_END__'
        if start_marker not in stdout or end_marker not in stdout:
            raise RuntimeError('Could not find subflow results in stdout.\nSTDOUT:\n' + stdout)
        payload = stdout.split(start_marker, 1)[1].split(end_marker, 1)[0].strip()
        if not payload:
            return {}
        return json.loads(payload)

    subflow_name = 'Workflow_1'
    subflow_nodes_df = pd.DataFrame([])
    subflow_connections_df = pd.DataFrame([])
    subflow_nodes_df = update_subflow(subflow_nodes_df)
    _results = run_subflow(subflow_name, subflow_nodes_df, subflow_connections_df, python_executable='C:/work/quest210-py313/Scripts/python.exe')
    return {
        'output': None,
    }